<a href="https://colab.research.google.com/github/asheta66/Capuchin-search-algorithm-CapSA-/blob/master/mealpy_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y numpy pandas scipy scikit-learn
!pip install numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.4.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.11.4
Uninstalling scipy-1.11.4:
  Successfully uninstalled scipy-1.11.4
Found existing installation: scikit-learn 1.4.2
Uninstalling scikit-learn-1.4.2:
  Successfully uninstalled scikit-learn-1.4.2
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached scipy-1.11.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached pandas-2.2.2-cp3

In [ ]:
#========================== IMPORTS ==========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

from mealpy import FloatVar

#========================== ALGORITHMS ==========================
from mealpy.evolutionary_based.GA import BaseGA
from mealpy.evolutionary_based.DE import OriginalDE

from mealpy.swarm_based.PSO import OriginalPSO
from mealpy.swarm_based.GWO import OriginalGWO
from mealpy.swarm_based.WOA import OriginalWOA
from mealpy.swarm_based.HHO import OriginalHHO
from mealpy.swarm_based.SSA import OriginalSSA

from mealpy.physics_based.WDO import OriginalWDO
from mealpy.physics_based.MVO import OriginalMVO
from mealpy.physics_based.EO import OriginalEO

from mealpy.human_based.TLO import OriginalTLO
from mealpy.human_based.BSO import OriginalBSO

from mealpy.bio_based.BBO import OriginalBBO
from mealpy.bio_based.SMA import OriginalSMA
from mealpy.bio_based.SOS import OriginalSOS

#========================== GLOBAL PARAMETERS ==========================
EPOCH = 80
POP_SIZE = 40


In [2]:


#========================== LOAD DATA ==========================
df = pd.read_csv("OSA_2023.csv").dropna()

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# encode labels (important for classification)
y = LabelEncoder().fit_transform(y)
y = y.reshape(-1, 1)

# scale inputs only
scaler_X = MinMaxScaler()
X = scaler_X.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


#========================== NN STRUCTURE ==========================
input_size = X_train.shape[1]
hidden_size = 10
output_size = len(np.unique(y))   # automatic for multi-class

weight_size = input_size*hidden_size + hidden_size + hidden_size*output_size + output_size


def unpack(weights):
    i = 0
    w1 = weights[i:i+input_size*hidden_size].reshape(input_size, hidden_size)
    i += input_size*hidden_size

    b1 = weights[i:i+hidden_size].reshape(1, hidden_size)
    i += hidden_size

    w2 = weights[i:i+hidden_size*output_size].reshape(hidden_size, output_size)
    i += hidden_size*output_size

    b2 = weights[i:].reshape(1, output_size)

    return w1, b1, w2, b2


def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def predict(X, weights):
    w1, b1, w2, b2 = unpack(weights)

    hidden = np.tanh(X @ w1 + b1)
    output = hidden @ w2 + b2

    if output_size == 1:
        return 1 / (1 + np.exp(-output))   # sigmoid
    else:
        return softmax(output)


#========================== FITNESS FUNCTION ==========================
def fitness(solution):
    y_pred = predict(X_train, solution)

    if output_size == 1:
        y_pred = np.clip(y_pred, 1e-8, 1-1e-8)
        return log_loss(y_train, y_pred)
    else:
        y_pred = np.clip(y_pred, 1e-8, 1-1e-8)
        return log_loss(y_train, y_pred)


problem = {
    "obj_func": fitness,
    "bounds": FloatVar(lb=(-1.,)*weight_size, ub=(1.,)*weight_size),
    "minmax": "min",
    "log_to": None
}


#========================== ALGORITHMS ==========================
algorithms = {
    "GA": BaseGA(epoch=EPOCH, pop_size=POP_SIZE),
    "DE": OriginalDE(epoch=EPOCH, pop_size=POP_SIZE),

    "PSO": OriginalPSO(epoch=EPOCH, pop_size=POP_SIZE),
    "GWO": OriginalGWO(epoch=EPOCH, pop_size=POP_SIZE),
    "WOA": OriginalWOA(epoch=EPOCH, pop_size=POP_SIZE),
    "HHO": OriginalHHO(epoch=EPOCH, pop_size=POP_SIZE),
    "SSA": OriginalSSA(epoch=EPOCH, pop_size=POP_SIZE),

    "WDO": OriginalWDO(epoch=EPOCH, pop_size=POP_SIZE),
    "MVO": OriginalMVO(epoch=EPOCH, pop_size=POP_SIZE),
    "EO": OriginalEO(epoch=EPOCH, pop_size=POP_SIZE),

    "TLO": OriginalTLO(epoch=EPOCH, pop_size=POP_SIZE),
    "BSO": OriginalBSO(epoch=EPOCH, pop_size=POP_SIZE),

    "BBO": OriginalBBO(epoch=EPOCH, pop_size=POP_SIZE),
    "SMA": OriginalSMA(epoch=EPOCH, pop_size=POP_SIZE),
    "SOS": OriginalSOS(epoch=EPOCH, pop_size=POP_SIZE),
}


#========================== METRICS ==========================
def classification_metrics(y_true, y_pred):
    if output_size == 1:
        y_label = (y_pred > 0.5).astype(int)
    else:
        y_label = np.argmax(y_pred, axis=1).reshape(-1, 1)

    acc = accuracy_score(y_true, y_label)
    prec = precision_score(y_true, y_label, average="macro")
    rec = recall_score(y_true, y_label, average="macro")
    f1 = f1_score(y_true, y_label, average="macro")

    return acc, prec, rec, f1


#========================== TRAIN ALL MODELS ==========================
results = []

for name, model in algorithms.items():

    print(f"\nRunning: {name}")

    gbest = model.solve(problem)
    best = gbest.solution

    y_train_pred = predict(X_train, best)
    y_test_pred = predict(X_test, best)

    train_acc, train_prec, train_rec, train_f1 = classification_metrics(y_train, y_train_pred)
    test_acc, test_prec, test_rec, test_f1 = classification_metrics(y_test, y_test_pred)

    results.append({
        "Algorithm": name,

        "Train_Acc": train_acc,
        "Train_Precision": train_prec,
        "Train_Recall": train_rec,
        "Train_F1": train_f1,

        "Test_Acc": test_acc,
        "Test_Precision": test_prec,
        "Test_Recall": test_rec,
        "Test_F1": test_f1
    })

    print(f"✔ {name} done | Test Acc = {test_acc:.4f}")


#========================== RESULTS TABLE ==========================
results_df = pd.DataFrame(results).sort_values(by="Test_Acc", ascending=False)

print("\n================ FINAL RESULTS ================")
display(results_df)


#========================== BEST MODEL ==========================
best_algo = results_df.iloc[0]["Algorithm"]
print("\nBest Algorithm:", best_algo)


#========================== BEST MODEL PLOT ==========================
best_model = algorithms[best_algo]
best_sol = best_model.g_best.solution

y_train_pred = predict(X_train, best_sol)
y_test_pred = predict(X_test, best_sol)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(y_train, label="Actual")
plt.plot((y_train_pred>0.5).astype(int), label="Predicted")
plt.title("Training Results")
plt.legend()
plt.grid()

plt.subplot(1,2,2)
plt.plot(y_test, label="Actual")
plt.plot((y_test_pred>0.5).astype(int), label="Predicted")
plt.title("Testing Results")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject